## 6. Machine Learning Supervisionato: Classificazione e Regressione Non-Lineare

Il presente modulo espande l'analisi predittiva implementando algoritmi della famiglia degli *Ensemble Trees*. A differenza della Cosine Similarity, i modelli di Machine Learning supervisionato analizzano la varianza del dataset per estrarre regole decisionali, assegnando pesi asimmetrici alle feature geometriche.

Verranno sviluppati due approcci complementari:
1. **Classificazione Binaria (Random Forest & XGBoost Classifier):** Per prevedere esclusivamente la direzione futura del mercato (Rialzista = 0, Ribassista = 1).
2. **Regressione Continua (XGBoost Regressor):** Per prevedere la magnitudo quantitativa (in punti/dollari) del movimento futuro. Questa stima è cruciale per filtrare i segnali deboli e garantire che il profitto atteso sia superiore ai costi di transazione (Spread).

### 6.1 Preparazione del Dataset e Split Cronologico (Target Multipli)
Per prevenire il *Future Leakage*, viene applicato uno split cronologico (80/20). La funzione di vettorizzazione è stata ingegnerizzata per estrarre simultaneamente le etichette binarie ($y_{class}$) e i valori continui ($y_{reg}$), garantendo un allineamento temporale perfetto tra le matrici.

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import accuracy_score, precision_score, classification_report, confusion_matrix, mean_absolute_error

"""
Obiettivo: Trasformare la serie storica sequenziale in un dataset supervisionato vettorializzato.
In breve: L'algoritmo estrae 4 feature spaziali (Body, Range, Upper/Lower Shadow) per ogni candela 
e le concatena in 'finestre' di 10 candele (40 feature totali). 
Infine, esegue uno split cronologico 80/20 per evitare il data leakage.
"""

def create_sliding_windows(df, window_size=10):
    df_features = df.copy()
    
    df_features['Body'] = df_features['Close'] - df_features['Open']
    df_features['Range'] = df_features['High'] - df_features['Low']
    df_features['Upper_Shadow'] = df_features['High'] - df_features[['Open', 'Close']].max(axis=1)
    df_features['Lower_Shadow'] = df_features[['Open', 'Close']].min(axis=1) - df_features['Low']
    
    target_class_array = np.where(df_features['Body'] >= 0, 0, 1)
    target_reg_array = df_features['Body'].values
    
    X, y_class, y_reg = [], [], []
    features_array = df_features[['Body', 'Range', 'Upper_Shadow', 'Lower_Shadow']].values
    
    for i in range(len(df_features) - window_size):
        window = features_array[i : i + window_size].flatten()
        X.append(window)
        y_class.append(target_class_array[i + window_size])
        y_reg.append(target_reg_array[i + window_size])
        
    return np.array(X), np.array(y_class), np.array(y_reg)

def print_classification_evaluation(model_name, y_true, y_pred, exec_time):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro')
    cm = confusion_matrix(y_true, y_pred)
    print(f"--- Modello: {model_name} ---")
    print(f"[TIME] Esecuzione: {exec_time:.2f} sec")
    print(f"[METRICS] Accuracy: {acc * 100:.2f}% | Precision (Macro): {prec * 100:.2f}%")
    print(f"[MATRIX] VR Corretti: {cm[0][0]} | Falsi Ribassi: {cm[0][1]} | Falsi Rialzi: {cm[1][0]} | V Rib Corretti: {cm[1][1]}\n")

# [INFO] Caricamento e Vettorizzazione
file_path = os.path.join(os.getcwd(), 'Data Management', 'ReadyData', 'XAUUSD_ReadyToUse.csv')
df = pd.read_csv(file_path, parse_dates=['Datetime'])
df_clean = df[df['Missing'] == False].copy().reset_index(drop=True)

X, y_class, y_reg = create_sliding_windows(df_clean, window_size=10)

# [INFO] Split Cronologico 80/20
split_index = int(len(X) * 0.8)
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y_class[:split_index], y_class[split_index:]
y_reg_train, y_reg_test = y_reg[:split_index], y_reg[split_index:]

print(f"[INFO] Training Set : {X_train.shape[0]} samples")
print(f"[INFO] Testing Set  : {X_test.shape[0]} samples")

[INFO] Training Set : 283624 samples
[INFO] Testing Set  : 70907 samples


### 6.2 Modelli di Classificazione Direzionale
Addestramento comparativo tra Random Forest e XGBoost per l'identificazione puramente binaria della polarità del mercato.

In [ ]:
print("[INFO] Addestramento Random Forest Classifier in corso...")
start_time = time.time()
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
print_classification_evaluation("Random Forest", y_test, y_pred_rf, time.time() - start_time)

print("[INFO] Addestramento XGBoost Classifier in corso...")
start_time = time.time()
xgb_class = XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1, eval_metric='logloss')
xgb_class.fit(X_train, y_train)
y_pred_xgb_c = xgb_class.predict(X_test)
print_classification_evaluation("XGBoost Classifier", y_test, y_pred_xgb_c, time.time() - start_time)

### 6.3 Modello di Regressione Quantitativa
Addestramento dell'algoritmo per la stima dell'ampiezza spaziale della candela successiva. Il modello minimizza l'errore quadratico, risultando intrinsecamente conservativo in presenza di elevato rumore di fondo. L'accuratezza direzionale viene derivata per via analitica (segno algebrico della stima) per consentire il confronto con i classificatori.

In [ ]:
print("[INFO] Addestramento XGBoost Regressor in corso...")
start_time = time.time()

xgb_reg = XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)
xgb_reg.fit(X_train, y_reg_train)
y_pred_reg = xgb_reg.predict(X_test)
exec_time_reg = time.time() - start_time

mae = mean_absolute_error(y_reg_test, y_pred_reg)
y_pred_reg_class_derived = np.where(y_pred_reg >= 0, 0, 1)
reg_directional_accuracy = accuracy_score(y_test, y_pred_reg_class_derived)

print(f"--- Modello: XGBoost Regressor ---")
print(f"[TIME] Esecuzione: {exec_time_reg:.2f} sec")
print(f"[METRICS] Mean Absolute Error (MAE): {mae:.4f} punti")
print(f"[METRICS] Accuracy Direzionale Derivata: {reg_directional_accuracy * 100:.2f}%\n")

### 6.4 Interpretazione Geometrica: Feature Importance
Il seguente modulo mappa l'importanza (Feature Importance) attribuita dal classificatore XGBoost alle singole componenti spaziali della Sliding Window. L'obiettivo è individuare quali elementi geometrici (es. l'ombra della candela 2 o il corpo della candela 10) veicolino il maggior peso decisionale.

In [ ]:
# Mappatura etichette feature
features_base = ['Body', 'Range', 'Upper Shadow', 'Lower Shadow']
feature_names = [f"Candela {i} - {feat}" for i in range(1, 11) for feat in features_base]

# Estrazione pesi dal classificatore XGBoost
importances = xgb_class.feature_importances_

df_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importanza': importances
})

df_top15 = df_importance.sort_values(by='Importanza', ascending=False).head(15)

# Plotting vettoriale
plt.figure(figsize=(10, 6))
plt.barh(df_top15['Feature'][::-1], df_top15['Importanza'][::-1], color='steelblue', edgecolor='black')
plt.title("XGBoost Classifier: Top 15 Feature Importance", fontsize=14, fontweight='bold')
plt.xlabel("Peso Relativo (Importanza)", fontsize=12)
plt.ylabel("Componente Geometrica", fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### 6.5 Analisi Esplorativa (EDA): Distribuzione delle Feature Spaziali
Procediamo all'ispezione della distribuzione delle feature estratte prima dell'addestramento. Ci aspettiamo una distribuzione a campana estremamente stretta, con picchi concentrati attorno allo zero, indicativa della natura rumorosa e contenuta dei movimenti a bassissima latenza (1 minuto).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 5))

# Estrazione sicura: se è un array NumPy si usano gli indici, se è Pandas si usano le colonne
if hasattr(X_train, 'columns'):
    data1, data2 = X_train.iloc[:, 0], X_train.iloc[:, 1]
    title1, title2 = f'Distribuzione: {X_train.columns[0]}', f'Distribuzione: {X_train.columns[1]}'
else:
    data1, data2 = X_train[:, 0], X_train[:, 1]
    title1, title2 = 'Distribuzione: Feature 1 (Body)', 'Distribuzione: Feature 2 (Range)'

plt.subplot(1, 2, 1)
sns.histplot(data1, bins=100, kde=True, color='steelblue')
plt.title(title1)
plt.xlabel('Valore (Punti)')
plt.ylabel('Frequenza')
plt.xlim(-3, 3)

plt.subplot(1, 2, 2)
sns.histplot(data2, bins=100, kde=True, color='steelblue')
plt.title(title2)
plt.xlabel('Valore (Punti)')
plt.ylabel('Frequenza')
plt.xlim(0, 5)

plt.tight_layout()
plt.show()

### 6.6 Estrazione delle Feature: Proiezione in Spazio Bidimensionale (PCA)
Per comprendere la separabilità lineare del dataset, applichiamo un algoritmo di riduzione della dimensionalità (Principal Component Analysis - PCA). Comprimendo le 36/40 feature spaziali in due singole componenti principali (PC1 e PC2), possiamo visualizzare uno *scatter plot* 2D. 

L'obiettivo è valutare se le candele rialziste (classe 0) e ribassiste (classe 1) formano cluster distinti. Nel contesto del mercato azionario ad alta frequenza, una sovrapposizione densa (*overlap*) giustificherebbe la difficoltà dei modelli nel superare la soglia di *Accuracy* del 50%.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_train)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_train, cmap='coolwarm', alpha=0.3, s=5)
plt.title('Proiezione PCA (2D) - Sovrapposizione delle Classi (Train Set)')
plt.xlabel(f'Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% invarianza)')
plt.ylabel(f'Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% invarianza)')
plt.legend(handles=scatter.legend_elements()[0], labels=['Rialzista (0)', 'Ribassista (1)'])
plt.grid(True, alpha=0.3)
plt.show()

### 6.7 Comparazione Visiva delle Metriche (Random Forest vs XGBoost)
Confrontiamo le metriche finali *Out-of-Sample* (*Accuracy* e *F1-Score Macro*) tramite un diagramma a barre. Questo grafico permette di visualizzare rapidamente come entrambi gli *Ensemble Trees* si comportino in modo quasi identico, scontrandosi con il limite intrinseco del *Random Walk* finanziario (evidenziato dalla soglia rossa al 50%).

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# Estrazione delle metriche chiave dai risultati delle predizioni
rf_acc = accuracy_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf, average='macro')
xgb_acc = accuracy_score(y_test, y_pred_xgb_c)
xgb_f1 = f1_score(y_test, y_pred_xgb_c, average='macro')

labels = ['Accuracy', 'F1-Score (Macro)']
rf_scores = [rf_acc, rf_f1]
xgb_scores = [xgb_acc, xgb_f1]

x = np.arange(len(labels))
width = 0.35

# Generazione del grafico a barre affiancate
plt.figure(figsize=(8, 6))
plt.bar(x - width/2, rf_scores, width, label='Random Forest', color='teal')
plt.bar(x + width/2, xgb_scores, width, label='XGBoost', color='darkorange')

plt.ylabel('Score')
plt.title('Confronto Prestazioni Modelli (Test Set)')
plt.xticks(x, labels)
plt.ylim(0.4, 0.6) # Zoom per visualizzare il delta minimale
plt.axhline(0.5, color='red', linestyle='--', linewidth=1.5, label='Random Walk (50%)')
plt.legend(loc='lower right')
plt.grid(axis='y', alpha=0.3)
plt.show()